# ASHRAE – Great Energy Predictor III · leaderboard notebook (Google Colab)

**What this notebook does**
1. Downloads the competition data with `kagglehub` (your snippet), plus the public leak dataset (optional).
2. Cleans the training data (site 0 electricity before 21 May 2016, zero electricity readings, flat-lines,
   out-of-season zero streaks) — the cleaning step was the main difference between the top solutions.
3. Aligns weather to local time, fills gaps and builds lag and rolling weather features.
4. Trains GPU **XGBoost** and **CatBoost** models (optionally CPU **LightGBM**), one per meter type, with
   month-block cross-validation plus a full-data refit.
5. Checks predictions against **leaked real 2017–2018 readings** (sites 0, 1, 2, 4, 15), tunes the ensemble
   weights and a per-meter scaling factor on them, and writes two submissions:
   *model only* and *model + leak insertion*.
6. Submits both to Kaggle and saves out-of-fold predictions for the assistant (notebook 02).

**About "top of the leaderboard":** the competition closed in December 2019. Late submissions are still
scored (public and private), but they are **not ranked**. For reference, the winning private score was
**1.231** (2nd: 1.232, 3rd: 1.234). In your portfolio, report the private score of *both* submissions,
because top teams also used leak insertion.

**Runtime:** use a GPU runtime (Runtime → Change runtime type → T4 GPU). With `High-RAM` it is
comfortable; on the free tier set `LOW_RAM = True`. A full run takes roughly 1.5–3 h; checkpoints are
saved to Google Drive, so you can re-run after a disconnect and it resumes.

**Before you start:** accept the competition rules on Kaggle, then add `KAGGLE_USERNAME` and `KAGGLE_KEY` to
Colab *Secrets* (🔑 icon in the left bar) — kagglehub and the Kaggle CLI read them.

In [ ]:
# ============================== CONFIG ==============================
COMPETITION = "ashrae-energy-prediction"
USE_LEAK = True                 # validate on + insert public leaked labels (sites 0,1,2,4,15)
LEAK_KERNEL = "yamsam/ashrae-leak-data-station"   # notebook output containing leak.feather
MODELS = ["xgb", "cat"]         # add "lgb" (CPU LightGBM, slower on Colab) or use ["hgb"] for a CPU-only run
N_FOLDS = 3                     # month-block folds: Jan–Apr / May–Aug / Sep–Dec
FULL_REFIT = True               # also train on all months with the CV-averaged number of rounds
FAST_DEV_RUN = False            # True = ~60 buildings, small test sample, no submission (pipeline check)
LOW_RAM = False                 # True = smaller test chunks, fewer cached arrays
SUBMIT = True                   # submit to Kaggle at the end
SEED = 42
SAVE_DIR = "/content/drive/MyDrive/ashrae_gepiii"   # checkpoints + submissions (falls back to /content)
DATA_DIR = None                 # set to a local folder to skip downloading (e.g. for testing)

In [ ]:
import os, sys, gc, glob, json, math, time, subprocess, warnings
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore")

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive, userdata
    try:
        drive.mount("/content/drive")
    except Exception as e:
        print("Drive not mounted (", e, ") — saving to /content instead")
        SAVE_DIR = "/content/ashrae_gepiii"
    for k in ("KAGGLE_USERNAME", "KAGGLE_KEY"):
        try:
            os.environ[k] = userdata.get(k)
        except Exception:
            print(f"Colab secret {k} not found — add it if downloads fail")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "catboost", "kagglehub", "kaggle"], check=False)
os.makedirs(SAVE_DIR, exist_ok=True)
CKPT = os.path.join(SAVE_DIR, "checkpoints"); os.makedirs(CKPT, exist_ok=True)

def gpu_available():
    try:
        return subprocess.run(["nvidia-smi"], capture_output=True).returncode == 0
    except FileNotFoundError:
        return False
GPU = gpu_available()
print("GPU:", GPU, "| save dir:", SAVE_DIR)

## 1 · Download data (kagglehub) and the leak file

In [ ]:
if DATA_DIR is None:
    import kagglehub
    # Download latest version
    path = kagglehub.competition_download(COMPETITION)
    print("Path to competition files:", path)
    DATA_DIR = path

LEAK_FILE = None
if USE_LEAK:
    leak_dir = None
    try:
        import kagglehub
        leak_dir = kagglehub.notebook_output_download(LEAK_KERNEL)
    except Exception as e:
        print("kagglehub.notebook_output_download failed:", repr(e)[:200], "→ trying the Kaggle CLI")
        leak_dir = "/content/leak"
        os.makedirs(leak_dir, exist_ok=True)
        subprocess.run(["kaggle", "kernels", "output", LEAK_KERNEL, "-p", leak_dir], check=False)
    cands = []
    for ext in ("feather", "parquet", "csv"):
        cands += glob.glob(os.path.join(str(leak_dir), "**", f"*leak*.{ext}"), recursive=True)
    LEAK_FILE = cands[0] if cands else None
    print("Leak file:", LEAK_FILE)
    if LEAK_FILE is None:
        print("⚠️ No leak file found — continuing without leak validation/insertion")
        USE_LEAK = False

## 2 · Load and shrink memory

In [ ]:
def load_competition(d):
    t0 = time.time()
    train = pd.read_csv(f"{d}/train.csv", parse_dates=["timestamp"],
                        dtype={"building_id": "int16", "meter": "int8", "meter_reading": "float32"})
    test = pd.read_csv(f"{d}/test.csv", parse_dates=["timestamp"],
                       dtype={"row_id": "int32", "building_id": "int16", "meter": "int8"})
    meta = pd.read_csv(f"{d}/building_metadata.csv")
    wtr = pd.read_csv(f"{d}/weather_train.csv", parse_dates=["timestamp"])
    wte = pd.read_csv(f"{d}/weather_test.csv", parse_dates=["timestamp"])
    print(f"loaded in {time.time() - t0:.0f}s: train {train.shape}, test {test.shape}")
    return train, test, meta, pd.concat([wtr, wte], ignore_index=True)

train, test, meta, weather = load_competition(DATA_DIR)
if FAST_DEV_RUN:
    keep = meta.groupby("site_id").building_id.apply(lambda s: s.head(4)).values
    train = train[train.building_id.isin(keep)]
    test = test[test.building_id.isin(keep)].sample(frac=0.25, random_state=SEED).sort_values("row_id")
    MODELS = [m for m in MODELS if m != "lgb"]
    print("FAST_DEV_RUN:", len(keep), "buildings")
test = test.reset_index(drop=True)

## 3 · Weather: UTC → local time, gap filling, derived features

In [ ]:
# Hours to subtract from weather timestamps (UTC) to get site-local time (public GEPIII kernels)
SITE_UTC_OFFSET = {0: 4, 1: 0, 2: 7, 3: 4, 4: 7, 5: 0, 6: 4, 7: 4, 8: 4, 9: 5, 10: 7, 11: 4, 12: 0, 13: 5, 14: 4, 15: 4}
WCOLS = ["air_temperature", "dew_temperature", "cloud_coverage", "sea_level_pressure",
         "wind_speed", "wind_direction", "precip_depth_1_hr"]

def prep_weather(w):
    w = w.copy()
    w["timestamp"] = w["timestamp"] - pd.to_timedelta(w.site_id.map(SITE_UTC_OFFSET).fillna(0), unit="h")
    grid = pd.MultiIndex.from_product(
        [sorted(w.site_id.unique()), pd.date_range("2016-01-01", "2018-12-31 23:00", freq="h")],
        names=["site_id", "timestamp"])
    w = (w.drop_duplicates(["site_id", "timestamp"]).set_index(["site_id", "timestamp"])
           .reindex(grid).reset_index().sort_values(["site_id", "timestamp"]))
    for c in WCOLS:
        w[c] = w.groupby("site_id")[c].transform(lambda s: s.interpolate(limit=24, limit_direction="both"))
        mh = w.groupby(["site_id", w.timestamp.dt.month, w.timestamp.dt.hour])[c].transform("mean")
        w[c] = w[c].fillna(mh)
        w[c] = w[c].fillna(w.groupby("site_id")[c].transform("mean")).fillna(w[c].mean())
    t, td = w.air_temperature, w.dew_temperature
    w["rh"] = 100 * np.exp(17.625 * td / (243.04 + td)) / np.exp(17.625 * t / (243.04 + t))
    g = w.groupby("site_id")["air_temperature"]
    for h in (24, 72, 168):
        w[f"air_temp_mean{h}"] = g.transform(lambda s: s.rolling(h, min_periods=1).mean())
    w["air_temp_lag3"] = g.shift(3).fillna(w.air_temperature)
    w["air_temp_diff24"] = w.air_temperature - w.air_temp_mean24
    w["dew_mean24"] = w.groupby("site_id")["dew_temperature"].transform(lambda s: s.rolling(24, min_periods=1).mean())
    day = w.timestamp.dt.floor("D")
    w["air_temp_day_max"] = w.groupby(["site_id", day]).air_temperature.transform("max")
    w["air_temp_day_min"] = w.groupby(["site_id", day]).air_temperature.transform("min")
    w["wind_dir_sin"] = np.sin(np.deg2rad(w.wind_direction)); w["wind_dir_cos"] = np.cos(np.deg2rad(w.wind_direction))
    w = w.drop(columns=["wind_direction"])
    for c in w.columns:
        if c not in ("site_id", "timestamp"):
            w[c] = w[c].astype("float32")
    w["site_id"] = w.site_id.astype("int8")
    return w

weather_f = prep_weather(weather); del weather; gc.collect()
print("weather features:", weather_f.shape)

## 4 · Clean the training data

In [ ]:
def clean_train(tr, meta):
    n0 = len(tr)
    tr = tr.merge(meta[["building_id", "site_id"]], on="building_id", how="left")
    tr = tr.sort_values(["building_id", "meter", "timestamp"]).reset_index(drop=True)
    bad = (tr.site_id == 0) & (tr.meter == 0) & (tr.timestamp < "2016-05-21")   # known site-0 issue
    bad |= (tr.meter == 0) & (tr.meter_reading <= 0)                             # zero electricity
    key = tr.building_id.astype("int32") * 4 + tr.meter
    new_run = (tr.meter_reading.diff() != 0) | (key.diff() != 0) | (tr.timestamp.diff() != pd.Timedelta("1h"))
    run_id = new_run.cumsum()
    run_len = run_id.map(run_id.value_counts())
    is_zero = tr.meter_reading == 0
    month = tr.timestamp.dt.month
    in_season = (tr.meter.isin([2, 3]) & month.between(5, 10)) | ((tr.meter == 1) & month.isin([11, 12, 1, 2, 3]))
    run_in_season = in_season.groupby(run_id).transform("all")
    bad |= (run_len >= 48) & ~is_zero                    # flat-lines (stuck / estimated readings)
    bad |= (run_len >= 48) & is_zero & ~run_in_season    # out-of-season zero streaks
    tr = tr[~bad].drop(columns=["site_id"]).reset_index(drop=True)
    print(f"cleaning removed {n0 - len(tr):,} rows ({100 * (n0 - len(tr)) / n0:.1f} %)")
    return tr

train = clean_train(train, meta); gc.collect()

## 5 · Features

In [ ]:
from pandas.tseries.holiday import USFederalHolidayCalendar
HOLIDAY_TS = pd.DatetimeIndex(USFederalHolidayCalendar().holidays(start="2016-01-01", end="2019-01-01"))

meta_f = meta.copy()
meta_f["primary_use"] = meta_f.primary_use.astype("category").cat.codes.astype("int8")
meta_f["square_feet_log"] = np.log1p(meta_f.square_feet).astype("float32")
meta_f = meta_f[["building_id", "site_id", "primary_use", "square_feet_log", "year_built", "floor_count"]]
meta_f["site_id"] = meta_f.site_id.astype("int8")

CAT_FEATS = ["building_id", "site_id", "primary_use"]
FEATS = CAT_FEATS + ["square_feet_log", "year_built", "floor_count", "hour", "weekday", "month", "is_holiday",
                     "air_temperature", "dew_temperature", "cloud_coverage", "sea_level_pressure", "wind_speed",
                     "precip_depth_1_hr", "wind_dir_sin", "wind_dir_cos", "rh", "air_temp_mean24", "air_temp_mean72",
                     "air_temp_mean168", "air_temp_lag3", "air_temp_diff24", "dew_mean24",
                     "air_temp_day_max", "air_temp_day_min"]

def make_features(df):
    df = df.merge(meta_f, on="building_id", how="left").merge(weather_f, on=["site_id", "timestamp"], how="left")
    ts = df.timestamp
    df["hour"] = ts.dt.hour.astype("int8"); df["weekday"] = ts.dt.weekday.astype("int8")
    df["month"] = ts.dt.month.astype("int8")
    df["is_holiday"] = ts.dt.normalize().isin(HOLIDAY_TS).astype("int8")
    return df

## 6 · Models (GPU XGBoost, GPU CatBoost, optional LightGBM / sklearn fallback)

In [ ]:
def model_params(name):
    if name == "xgb":
        return dict(n_estimators=4000, learning_rate=0.05, max_depth=12, min_child_weight=5, subsample=0.8,
                    colsample_bytree=0.7, reg_lambda=2.0, tree_method="hist", device="cuda" if GPU else "cpu",
                    max_bin=256, enable_categorical=True, max_cat_to_onehot=1, early_stopping_rounds=100,
                    eval_metric="rmse", random_state=SEED)
    if name == "cat":
        return dict(iterations=4000, learning_rate=0.1, depth=10, l2_leaf_reg=3, loss_function="RMSE",
                    task_type="GPU" if GPU else "CPU", border_count=254, random_seed=SEED, od_type="Iter",
                    od_wait=100, verbose=500)
    if name == "lgb":
        return dict(objective="regression", n_estimators=4000, learning_rate=0.05, num_leaves=1023,
                    feature_fraction=0.8, bagging_fraction=0.8, bagging_freq=1, lambda_l2=2, min_data_in_leaf=50,
                    max_bin=255, seed=SEED, verbose=-1)
    if name == "hgb":
        return dict(max_iter=600, learning_rate=0.1, max_leaf_nodes=255, l2_regularization=1.0,
                    early_stopping=False, random_state=SEED)
    raise ValueError(name)

# Fixed category levels so train and test frames share the same codes (XGBoost uses the codes)
CAT_LEVELS = {"building_id": sorted(meta.building_id.unique()), "site_id": sorted(meta.site_id.unique()),
              "primary_use": sorted(meta_f.primary_use.unique())}

def as_model_frame(X, name):
    X = X[FEATS].copy()
    if name == "xgb":
        for c in CAT_FEATS:
            X[c] = pd.Categorical(X[c], categories=CAT_LEVELS[c])
    if name == "cat":
        for c in CAT_FEATS:
            X[c] = X[c].astype("int32")
    return X

def fit_model(name, Xtr, ytr, Xva=None, yva=None, n_rounds=None):
    p = model_params(name)
    if name == "xgb":
        import xgboost as xgb
        if n_rounds is not None:
            p.update(n_estimators=n_rounds); p.pop("early_stopping_rounds")
        m = xgb.XGBRegressor(**p)
        m.fit(as_model_frame(Xtr, name), ytr,
              eval_set=[(as_model_frame(Xva, name), yva)] if Xva is not None else None, verbose=500)
        best = getattr(m, "best_iteration", None)
        return m, (best + 1) if best is not None else p["n_estimators"]
    if name == "cat":
        from catboost import CatBoostRegressor, Pool
        if n_rounds is not None:
            p.update(iterations=n_rounds); p.pop("od_type"); p.pop("od_wait")
        m = CatBoostRegressor(**p)
        m.fit(Pool(as_model_frame(Xtr, name), ytr, cat_features=CAT_FEATS),
              eval_set=Pool(as_model_frame(Xva, name), yva, cat_features=CAT_FEATS) if Xva is not None else None,
              use_best_model=Xva is not None)
        return m, (m.get_best_iteration() or p["iterations"]) + 1
    if name == "lgb":
        import lightgbm as lgb
        if n_rounds is not None:
            p.update(n_estimators=n_rounds)
        m = lgb.LGBMRegressor(**p)
        cb = [lgb.early_stopping(100), lgb.log_evaluation(500)] if Xva is not None and n_rounds is None else []
        m.fit(as_model_frame(Xtr, name), ytr, eval_set=[(as_model_frame(Xva, name), yva)] if Xva is not None else None,
              categorical_feature=CAT_FEATS, callbacks=cb)
        return m, (m.best_iteration_ or p["n_estimators"])
    if name == "hgb":
        from sklearn.ensemble import HistGradientBoostingRegressor
        if n_rounds is not None:
            p.update(max_iter=n_rounds)
        m = HistGradientBoostingRegressor(**p, categorical_features=[FEATS.index(c) for c in ("site_id", "primary_use")])
        m.fit(as_model_frame(Xtr, name).fillna(-999).values, ytr)
        return m, p["max_iter"]

def predict_model(name, m, X):
    Xm = as_model_frame(X, name)
    if name == "hgb":
        Xm = Xm.fillna(-999).values
    return np.asarray(m.predict(Xm), dtype="float32")

def rmsle_log(y_log, p_log):
    return float(np.sqrt(np.mean((np.asarray(y_log) - np.asarray(p_log)) ** 2)))

## 7 · Train: month-block CV per meter + full refit (with Drive checkpoints)

In [ ]:
FOLD_MONTHS = {3: [[1, 2, 3, 4], [5, 6, 7, 8], [9, 10, 11, 12]],
               2: [[1, 2, 3, 4, 5, 6], [7, 8, 9, 10, 11, 12]],
               4: [[1, 2, 3], [4, 5, 6], [7, 8, 9], [10, 11, 12]]}[N_FOLDS]
CHUNK = 2_000_000 if LOW_RAM else 5_000_000

def ckpt_path(*parts):
    return os.path.join(CKPT, "_".join(str(p) for p in parts) + ".npy")

cv_rows, oof_store = [], {}
test_pred = {name: np.zeros(len(test), dtype="float32") for name in MODELS}
test_index = {mt: np.where(test.meter.values == mt)[0] for mt in range(4)}

for mt in range(4):
    tr_m = train[train.meter == mt]
    if tr_m.empty:
        continue
    X = make_features(tr_m.drop(columns=["meter_reading"]))
    y = np.log1p(tr_m.meter_reading.values).astype("float32")
    months = tr_m.timestamp.dt.month.values
    te_idx = test_index[mt]
    for name in MODELS:
        oof = np.full(len(y), np.nan, dtype="float32")
        rounds = []
        fold_test = np.zeros(len(te_idx), dtype="float32")
        n_parts = 0
        for k, va_months in enumerate(FOLD_MONTHS):
            f_oof, f_te = ckpt_path(name, mt, k, "oof"), ckpt_path(name, mt, k, "test")
            va = np.isin(months, va_months)
            if os.path.exists(f_oof) and os.path.exists(f_te):
                oof[va] = np.load(f_oof); fold_test += np.load(f_te); n_parts += 1
                rounds.append(int(json.load(open(f_oof + ".json"))["rounds"]))
                print(f"[resume] {name} meter {mt} fold {k}")
                continue
            t0 = time.time()
            m, best = fit_model(name, X[~va], y[~va], X[va], y[va])
            oof[va] = predict_model(name, m, X[va])
            pt = np.zeros(len(te_idx), dtype="float32")
            for s in range(0, len(te_idx), CHUNK):
                part = make_features(test.iloc[te_idx[s:s + CHUNK]])
                pt[s:s + CHUNK] = predict_model(name, m, part)
                del part; gc.collect()
            np.save(f_oof, oof[va]); np.save(f_te, pt)
            json.dump({"rounds": int(best)}, open(f_oof + ".json", "w"))
            fold_test += pt; n_parts += 1; rounds.append(int(best))
            print(f"{name} meter {mt} fold {k}: rmse {rmsle_log(y[va], oof[va]):.4f}  rounds {best}  {time.time() - t0:.0f}s")
            del m; gc.collect()
        fold_test /= max(n_parts, 1)
        final = fold_test
        if FULL_REFIT:
            f_full = ckpt_path(name, mt, "full", "test")
            if os.path.exists(f_full):
                full_pred = np.load(f_full)
            else:
                n_r = int(np.mean(rounds) * 1.1) if rounds else None
                m, _ = fit_model(name, X, y, n_rounds=n_r)
                full_pred = np.zeros(len(te_idx), dtype="float32")
                for s in range(0, len(te_idx), CHUNK):
                    part = make_features(test.iloc[te_idx[s:s + CHUNK]])
                    full_pred[s:s + CHUNK] = predict_model(name, m, part)
                    del part; gc.collect()
                np.save(f_full, full_pred)
                if name in ("xgb", "lgb", "cat") and mt == 0:
                    try:
                        imp = pd.Series(m.feature_importances_, index=FEATS).sort_values(ascending=False)
                        imp.to_csv(os.path.join(SAVE_DIR, f"feature_importance_{name}_meter0.csv"))
                    except Exception:
                        pass
                del m; gc.collect()
            final = 0.5 * fold_test + 0.5 * full_pred
        test_pred[name][te_idx] = final
        oof_store[(name, mt)] = oof
        cv_rows.append({"model": name, "meter": mt, "rows": len(y), "cv_rmsle": rmsle_log(y, oof),
                        "rounds_mean": float(np.mean(rounds)) if rounds else None})
        print(cv_rows[-1])
    if not LOW_RAM:
        pd.DataFrame({"building_id": tr_m.building_id.values, "meter": mt, "timestamp": tr_m.timestamp.values,
                      **{f"oof_{n}": oof_store[(n, mt)] for n in MODELS}, "y_log": y}).to_parquet(
            os.path.join(SAVE_DIR, f"oof_meter{mt}.parquet"), index=False)
    del X, y, tr_m; gc.collect()

cv = pd.DataFrame(cv_rows)
display(cv) if "display" in globals() else print(cv)

## 8 · Validate on leaked 2017–2018 readings, blend, calibrate

In [ ]:
def load_leak(fp):
    if fp is None:
        return None
    lk = pd.read_feather(fp) if fp.endswith(".feather") else pd.read_parquet(fp) if fp.endswith(".parquet") else pd.read_csv(fp)
    lk.columns = [c.lower() for c in lk.columns]
    lk["timestamp"] = pd.to_datetime(lk["timestamp"])
    lk = lk.dropna(subset=["meter_reading"])
    lk = lk[lk.meter_reading >= 0]
    return lk[["building_id", "meter", "timestamp", "meter_reading"]]

leak_y = np.full(len(test), np.nan, dtype="float32")
if USE_LEAK:
    lk = load_leak(LEAK_FILE)
    # keep only (building, meter) pairs whose 2016 leak readings match the official train data
    lk16 = lk[lk.timestamp < "2017-01-01"]
    if len(lk16):
        chk = lk16.merge(train[["building_id", "meter", "timestamp", "meter_reading"]],
                         on=["building_id", "meter", "timestamp"], suffixes=("_leak", ""))
        err = chk.assign(e=(np.log1p(chk.meter_reading_leak) - np.log1p(chk.meter_reading)) ** 2) \
                 .groupby(["building_id", "meter"]).e.mean().pow(0.5)
        good_pairs = set(err[err < 0.1].index)
        print(f"leak pairs matching 2016 train: {len(good_pairs)} / {err.shape[0]}")
    else:
        good_pairs = None
    lk = lk[lk.timestamp >= "2017-01-01"]
    if good_pairs is not None:
        lk = lk[[(b, m) in good_pairs for b, m in zip(lk.building_id, lk.meter)]]
    joined = test[["row_id", "building_id", "meter", "timestamp"]].reset_index().merge(
        lk, on=["building_id", "meter", "timestamp"], how="inner")
    leak_y[joined["index"].values] = joined.meter_reading.values
    print(f"test rows with leaked labels: {np.isfinite(leak_y).sum():,} ({100 * np.isfinite(leak_y).mean():.1f} %)")

has_leak = np.isfinite(leak_y)
weights, alphas, report = {}, {}, []
for mt in range(4):
    idx = test_index[mt]
    if len(idx) == 0:
        continue
    names = MODELS
    if USE_LEAK and has_leak[idx].sum() > 1000:
        yl = np.log1p(leak_y[idx][has_leak[idx]])
        P = np.column_stack([test_pred[n][idx][has_leak[idx]] for n in names])
        source = "leak"
    else:  # fall back to out-of-fold predictions
        mask = np.isfinite(oof_store[(names[0], mt)])
        yl = None
        P = np.column_stack([oof_store[(n, mt)] for n in names])
        tr_y = np.log1p(train[train.meter == mt].meter_reading.values)
        yl, P = tr_y[mask], P[mask]
        source = "oof"
    best = (1e9, None)
    grid = [np.array(w) for w in np.ndindex(*([11] * len(names))) if sum(w) == 10] if len(names) > 1 else [np.array([10])]
    for w in grid:
        w = w / 10.0
        s = rmsle_log(yl, P @ w)
        if s < best[0]:
            best = (s, w)
    w = best[1]
    blend = P @ w
    a_best = (rmsle_log(yl, blend), 1.0)
    if source == "leak":
        for a in np.arange(0.90, 1.051, 0.005):     # scale in raw space ≈ shift in log space
            s = rmsle_log(yl, np.log1p(np.expm1(blend) * a))
            if s < a_best[0]:
                a_best = (s, float(a))
    weights[mt], alphas[mt] = dict(zip(names, w.round(2))), a_best[1]
    row = {"meter": mt, "validated_on": source, "rows": int(len(yl)),
           **{f"rmsle_{n}": rmsle_log(yl, P[:, i]) for i, n in enumerate(names)},
           "rmsle_blend": best[0], "scale": a_best[1], "rmsle_blend_scaled": a_best[0], "weights": weights[mt]}
    report.append(row); print(row)

val_report = pd.DataFrame(report)
val_report.to_csv(os.path.join(SAVE_DIR, "validation_report.csv"), index=False)
cv.to_csv(os.path.join(SAVE_DIR, "cv_report.csv"), index=False)

## 9 · Build submissions (model only, and model + leak insertion)

In [ ]:
final_log = np.zeros(len(test), dtype="float32")
for mt in range(4):
    idx = test_index[mt]
    if len(idx) == 0:
        continue
    blend = sum(weights[mt][n] * test_pred[n][idx] for n in MODELS)
    final_log[idx] = blend
pred = np.expm1(final_log)
for mt in range(4):
    pred[test_index[mt]] *= alphas.get(mt, 1.0)
pred = np.clip(pred, 0, None).astype("float32")

sub_model = pd.DataFrame({"row_id": test.row_id.values, "meter_reading": pred})
files = {"model_only": os.path.join(SAVE_DIR, "submission_model_only.csv.gz")}
sub_model.sort_values("row_id").to_csv(files["model_only"], index=False, float_format="%.4f")
if USE_LEAK and has_leak.any():
    sub_leak = sub_model.copy()
    sub_leak.loc[has_leak, "meter_reading"] = leak_y[has_leak]
    files["with_leak"] = os.path.join(SAVE_DIR, "submission_with_leak.csv.gz")
    sub_leak.sort_values("row_id").to_csv(files["with_leak"], index=False, float_format="%.4f")
    print("leak-validated RMSLE of the final blend (rows with leak):",
          rmsle_log(np.log1p(leak_y[has_leak]), np.log1p(pred[has_leak])))
print(files)

## 10 · Submit to Kaggle and fetch scores

In [ ]:
if SUBMIT and not FAST_DEV_RUN:
    tag = "+".join(MODELS)
    for kind, fp in files.items():
        msg = f"{tag} {N_FOLDS}fold month-CV, cleaned, tz-aligned, {kind}"
        subprocess.run(["kaggle", "competitions", "submit", "-c", COMPETITION, "-f", fp, "-m", msg], check=False)
    time.sleep(60)
    print(subprocess.run(["kaggle", "competitions", "submissions", "-c", COMPETITION],
                         capture_output=True, text=True).stdout[:3000])
    print("Scoring takes a few minutes. Re-run this line later to see public/private scores:")
    print("  !kaggle competitions submissions -c", COMPETITION)

## 11 · Portfolio summary (copy into your README)

In [ ]:
summary = {
    "models": MODELS, "folds": N_FOLDS, "full_refit": FULL_REFIT,
    "cv_rmsle_by_meter": cv.groupby("meter").cv_rmsle.min().round(4).to_dict(),
    "validation": val_report.drop(columns=["weights"]).round(4).to_dict(orient="records"),
    "weights": {int(k): {n: float(v) for n, v in w.items()} for k, w in weights.items()},
    "scale_factors": {int(k): float(v) for k, v in alphas.items()},
    "leak_rows": int(has_leak.sum()), "submissions": files,
}
json.dump(summary, open(os.path.join(SAVE_DIR, "model_summary.json"), "w"), indent=2, default=str)
print(json.dumps(summary, indent=2, default=str)[:4000])